In [1]:
import math
import numpy as np
import random
from collections import defaultdict

class TicTacToe:
    def __init__(self):
        self.board = [0] * 9  # 0 = empty, 1 = X, -1 = O
        self.current_player = 1

    def clone(self):
        clone = TicTacToe()
        clone.board = self.board[:]
        clone.current_player = self.current_player
        return clone

    def get_legal_moves(self):
        return [i for i in range(9) if self.board[i] == 0]

    def make_move(self, move):
        self.board[move] = self.current_player
        self.current_player *= -1

    def game_over(self):
        lines = [(0,1,2), (3,4,5), (6,7,8),
                 (0,3,6), (1,4,7), (2,5,8),
                 (0,4,8), (2,4,6)]
        for a,b,c in lines:
            if self.board[a] != 0 and self.board[a] == self.board[b] == self.board[c]:
                return self.board[a]  # winner
        if 0 not in self.board:
            return 0  # draw
        return None  # not over

class DummyNN:
    def predict(self, state):
        legal_moves = state.get_legal_moves()
        policy = np.zeros(9)
        for move in legal_moves:
            policy[move] = 1.0
        policy /= policy.sum()
        value = 0  # pretend value
        return policy, value

class MCTSNode:
    def __init__(self, state, parent=None):
        self.state = state
        self.parent = parent
        self.children = {}
        self.visit_count = 0
        self.total_value = 0
        self.prior = 0

    def is_expanded(self):
        return len(self.children) > 0

    def value(self):
        if self.visit_count == 0:
            return 0
        return self.total_value / self.visit_count

    def ucb_score(self, c_puct=1.0):
        if self.parent is None:
            return self.value()
        return self.value() + c_puct * self.prior * math.sqrt(self.parent.visit_count) / (1 + self.visit_count)

class MCTS:
    def __init__(self, nn, simulations=50):
        self.nn = nn
        self.simulations = simulations

    def run(self, root):
        for _ in range(self.simulations):
            node = root
            search_path = [node]

            # Selection
            while node.is_expanded() and node.state.game_over() is None:
                max_ucb = max(node.children.values(), key=lambda n: n.ucb_score())
                node = max_ucb
                search_path.append(node)

            # Expansion
            result = node.state.game_over()
            if result is None:
                policy, value = self.nn.predict(node.state)
                for move in node.state.get_legal_moves():
                    next_state = node.state.clone()
                    next_state.make_move(move)
                    child = MCTSNode(next_state, parent=node)
                    child.prior = policy[move]
                    node.children[move] = child
            else:
                value = result * node.state.current_player  # From current player’s view

            # Backpropagation
            for node in reversed(search_path):
                node.visit_count += 1
                node.total_value += value
                value